# Repository edition: 08_overnight

Run from this repository in your existing GPU environment. Source is in `../suites/08_overnight/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Prioritized overnight experiments
Self-contained code; **no installation or Hugging Face token cells**. Uses your working GPU Python.

GPU order: **1B nulls → 1B singleton freezes → Qwen3B split → OLMo7B split → optional 1B matrix**.
Each job gets a fresh run name. Full cohort by default except the 100-prompt matrix. Runtime has not been benchmarked; the nulls are not guaranteed to finish within an hour.

CPU cache analysis starts alongside the GPU queue. Older caches support origin-input ANOVA and scalar norm variation, **not actual gate ANOVA or first-token vector constancy**. Missing inputs are reported. No previous results are required for the GPU jobs.


In [ ]:
from pathlib import Path
import ast,base64,hashlib,io,json,os,subprocess,sys,time,uuid,zipfile
BUNDLE_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('08_overnight'))).hexdigest()
BUNDLE = _source_bundle('08_overnight')

payload=base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest()==BUNDLE_SHA256
CODE=Path.home()/('threeways_overnight_code_'+BUNDLE_SHA256[:12])
CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        if CODE.resolve() not in (CODE/item.filename).resolve().parents:raise ValueError('Unsafe path')
    z.extractall(CODE)
print('Code:',CODE)


## 1. Choose your existing Python

In [ ]:
candidates=[Path('/home/ubuntu/venvs/threeways/bin/python'),Path(sys.executable),Path.home()/'threeways_causal_env/bin/python']
check="import torch,transformers,numpy,pandas,matplotlib,datasets,huggingface_hub; assert transformers.__version__=='4.48.3', 'Expected tested Transformers 4.48.3'; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name())"
PYTHON=None;errors=[]
for candidate in dict.fromkeys(candidates):
    if not candidate.exists():continue
    p=subprocess.run([str(candidate),'-c',check],capture_output=True,text=True)
    if p.returncode==0:
        PYTHON=candidate;print('Using:',candidate,'|',p.stdout.strip());break
    errors.append(str(candidate)+': '+p.stderr[-1200:])
if PYTHON is None:raise RuntimeError('Point candidates to your working environment.\n'+'\n'.join(errors))


## 2. Create a new campaign
Keep `PREVIOUS=None` if the 7 GB archive is only on your computer. To run CPU cache analyses, point it to an **extracted old result folder on this GPU**. Uploading the entire archive is optional.

`None` below means the full cohort, not a smoke test. Set the two limits to `100` for a prespecified smaller study if time is limited. All jobs share one cohort and deterministic subset ordering. The default random-readout seed is fixed across prompts.

The free-disk floor is not a download-size estimate; model downloads can require additional tens of GiB.


In [ ]:
from datetime import datetime
import shutil
PREVIOUS=None
# PREVIOUS=Path('/home/ubuntu/threeways_channel_results/channels_fresh_v1')
NULL_PROMPTS=None
FREEZE_PROMPTS=None
RUN_MATRIX=True
MINIMUM_FREE_GIB=20

if PREVIOUS is not None:
    PREVIOUS=Path(PREVIOUS).expanduser().resolve()
    if not PREVIOUS.is_dir():raise ValueError('PREVIOUS must be an extracted results directory, or None.')
CAMPAIGN_NAME='overnight_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RESULTS=Path.home()/'threeways_overnight_results'/CAMPAIGN_NAME
RESULTS.mkdir(parents=True,exist_ok=False)
CONFIG=json.loads((CODE/'followup_study/config.json').read_text())
CONFIG.update(null_prompts=NULL_PROMPTS,freeze_prompts=FREEZE_PROMPTS,
              null_models=['llama1b'],freeze_models=['llama1b'],matrix_models=['llama1b'],
              matrix_prompts=100,readout_seeds=[0],unit_readout=True,freeze_mode='singletons')
JOBS=[
    dict(name='01_llama1b_nulls_'+CAMPAIGN_NAME,models=['llama1b'],stages=['nulls']),
    dict(name='02_llama1b_freezes_'+CAMPAIGN_NAME,models=['llama1b'],stages=['freeze']),
    dict(name='03_qwen3b_split_'+CAMPAIGN_NAME,models=['qwen3b'],stages=['profiles']),
    dict(name='04_olmo7b_split_'+CAMPAIGN_NAME,models=['olmo7b'],stages=['profiles']),
]
if RUN_MATRIX:JOBS.append(dict(name='05_llama1b_matrix_'+CAMPAIGN_NAME,models=['llama1b'],stages=['matrix']))
PLAN=dict(root=str(RESULTS),previous=str(PREVIOUS) if PREVIOUS else None,config=CONFIG,jobs=JOBS,minimum_free_gib=MINIMUM_FREE_GIB)
PLAN_PATH=RESULTS/'plan.json';PLAN_PATH.write_text(json.dumps(PLAN,indent=2))
print('Fresh campaign:',RESULTS)
print('Free disk GiB:',round(shutil.disk_usage(RESULTS).free/2**30,1))
print('Null prompts:',NULL_PROMPTS or 'ALL','; freeze prompts:',FREEZE_PROMPTS or 'ALL')
print('Backward passes per prompt: nulls 6; freezes 5; split 2; 1B matrix 18.')
print('These exclude captures, generation, gradient statistics and reporting.')
for job in JOBS:print(job['name'],job['stages'])


## 3. Launch once
The worker runs independently of this notebook. Keep the GPU instance powered on. Rerunning this cell cannot resume or relaunch an already-started campaign. If you deliberately want a new campaign, rerun configuration first.


In [ ]:
import fcntl
if 'queue_process' in globals() and queue_process.poll() is None:raise RuntimeError('A queue launched by this notebook is still running; use monitor.')
if (RESULTS/'started.json').exists():raise RuntimeError('Already launched. Use monitor, or create a new campaign deliberately.')
with (RESULTS/'queue.lock').open('a+') as lock:
    try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:raise RuntimeError('Queue is running.')
with (RESULTS/'queue.log').open('a') as log:
    queue_process=subprocess.Popen([str(PYTHON),'-u','-m','followup_study.overnight','--plan',str(PLAN_PATH)],
        cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('Queue PID:',queue_process.pid)
print('Results:',RESULTS)


## 4. Monitor without reading thousands of files

In [ ]:
def tail(path,limit=7000):
    if not path.exists():return
    with path.open('rb') as f:
        f.seek(max(0,path.stat().st_size-limit));print(f.read().decode(errors='replace'))
tail(RESULTS/'queue.log')
if (RESULTS/'active.json').exists():
    active=json.loads((RESULTS/'active.json').read_text());print('Most recent GPU job:',active['job']);tail(Path(active['log']))
tail(RESULTS/'cpu.log',2000)
if (RESULTS/'exit.json').exists():print('Queue exit:',json.loads((RESULTS/'exit.json').read_text()))
print('Free disk GiB:',round(shutil.disk_usage(RESULTS).free/2**30,1))


## 5. Inspect completed summaries
Each GPU job creates its own reports after finishing. A successful queue has `failed=false`, `stopped=false`, and one zero-return-code entry for every requested job. CPU `status.json` lists which inputs were available. Cache analyses are descriptive if the old run was partial.


In [ ]:
import csv
for job in JOBS:
    root=RESULTS/job['name']
    for name in ['null_paired_summary.csv','freeze_summary.csv','profile_summary.csv','matrix_summary.csv']:
        p=root/'reports'/name
        if p.exists():
            print('\n',job['name'],name)
            with p.open() as f:
                for i,row in enumerate(csv.DictReader(f)):
                    if i==4:break
                    print({k:v for k,v in row.items() if not k.endswith(('_lo','_hi','_valid_facts'))})
p=RESULTS/'00_cached_cpu/status.json'
if p.exists():print('\nCPU cache status:',json.loads(p.read_text()))


## 6. Stop cooperatively (only if wanted)
Uncomment the line below. The current operation finishes before stopping; wait for queue exit before powering down. This queue intentionally has no automatic resumption. Saved partial results remain available for inspection/export.


In [ ]:
# (RESULTS/'STOP').touch()


## 7. Small review export
Includes summaries, figures, cohort, configuration, model revisions and status. Excludes raw prompt measurements and statistic states; **not a full reanalysis/resume backup**. Download before terminating the instance. Run after jobs finish for a complete export.


In [ ]:
from IPython.display import display,FileLink
ARCHIVE=RESULTS.parent/(RESULTS.name+'_summaries.zip')
files=set()
for pattern in ['*/reports/*.csv','*/reports/*.pdf','*/reports/*.json','00_cached_cpu*/*.csv','00_cached_cpu*/status.json',
                '*/configuration.json','*/protocol.json','*/environment.json','*/completion.json','*/exit.json',
                '*/*/model.json','*/*/architecture.json','cohort_source/data/cohort.json']:
    files.update(RESULTS.glob(pattern))
for name in ['plan.json','started.json','progress.json','exit.json','error.json']:
    p=RESULTS/name
    if p.exists():files.add(p)
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(files):z.write(p,p.relative_to(RESULTS))
with zipfile.ZipFile(ARCHIVE) as z:assert z.testzip() is None
print(f'{len(files)} files; {ARCHIVE.stat().st_size/2**20:.2f} MiB; {ARCHIVE}')
display(FileLink(os.path.relpath(ARCHIVE,Path.cwd())))


## Optional: missing gate/sink captures
**Do not run alongside the GPU queue.** This is extra GPU work, not CPU postprocessing. Use only if you want actual gate/attention ANOVA and first-token vector constancy and the cache status says those inputs are missing. It reuses the campaign cohort, uses templates 0/1/2, and streams compact sufficient statistics. After completion, rerun the CPU analyzer against this new result folder to get the sink ratio table. The original reviewer notebook can also run these stages.


In [ ]:
# Uncomment only after the GPU queue exits, if these missing measurements are needed.
# if not (RESULTS/'exit.json').exists():raise RuntimeError('Wait for the queue to exit first.')
# EXTRA=RESULTS/('extra_gate_sink_'+uuid.uuid4().hex[:8]);EXTRA.mkdir()
# extra_config=dict(CONFIG)
# extra_config['models']=[m for m in CONFIG['models'] if m['key'] in ['llama1b','llama3b','llama8b','qwen7b']]
# extra_config['templates']=[0,1,2]
# cohort=json.loads((RESULTS/'cohort_source/data/cohort.json').read_text())
# cohort['evaluation']=[r for r in cohort['evaluation'] if r['template'] in [0,1,2]]
# (EXTRA/'data').mkdir();(EXTRA/'data/cohort.json').write_text(json.dumps(cohort))
# (EXTRA/'configuration.json').write_text(json.dumps(extra_config,indent=2))
# with (EXTRA/'supervisor.log').open('a') as log:
#     extra_worker=subprocess.Popen([str(PYTHON),'-u','-m','followup_study.supervisor','--config',str(EXTRA/'configuration.json'),
#          '--out',str(EXTRA),'--stages','variance','sink'],cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
# print('Extra captures:',EXTRA)

# AFTER extra_worker finishes, run:
# subprocess.run([str(PYTHON),'-m','followup_study.cached_cpu','--previous',str(EXTRA),
#                 '--out',str(RESULTS/'00_cached_cpu_recaptured')],cwd=CODE,check=True)
# Then rerun the export cell; recaptured CPU summaries are included automatically.


## Optional numerical tests
Tiny random CPU models; no downloads. These validate the implementation, not performance on a pretrained GH200 run.


In [ ]:
# subprocess.run([str(PYTHON),'-m','unittest','discover','-s','tests','-v'],cwd=CODE,check=True)
